# 05 — DPO and improvement data: from "caught it" to "trained on it"

## The training ladder, from zero
How a chat/voice model comes to behave:
1. **Pretraining** — next-token prediction over the internet. Capability, no manners.
2. **SFT** (supervised fine-tuning) — show it curated example conversations; it imitates. Format + style.
3. **Preference tuning** — show it PAIRS: same prompt, a **chosen** response and a **rejected** one; train it to prefer chosen-like behavior. This is where "do not talk over callers" type behavior actually gets installed.

Step 3's classic recipe was **RLHF**: train a separate *reward model* on the pairs, then run reinforcement learning against it. Powerful, notoriously fiddly. **DPO — Direct Preference Optimization (2023)** — collapsed it: a bit of algebra shows you can skip the reward model and update directly on the pairs. Intuition in one line: **raise the model's log-probability of the chosen response relative to the rejected one, while staying anchored to a frozen reference copy so the model does not drift into nonsense** (a beta knob controls the leash). No RL loop, one dataset format, standard tooling.

That dataset format is the punchline of this whole project: **every failure VoiceForge detects can be emitted as one (prompt, chosen, rejected) line.** Evals usually end at a dashboard. Training data is what the dashboard *cannot* do.

## The format (TRL = HuggingFace's training library that eats this directly)
One JSON object per line ("JSONL"):
- `prompt` — conversation up to the failure moment (system + prior turns)
- `rejected` — what the agent ACTUALLY said (we have it; it is in the transcript)
- `chosen` — the corrected turn

## The single-axis rule (our discipline, your ablation instinct)
Chosen and rejected must differ **only on the detected failure axis** — barged in → shorter acknowledging turn; ignored hesitation → wait + clarify; over-demanded → confirm partial + one follow-up. If chosen is also more polite, better formatted, and longer, the gradient cannot tell WHICH improvement it is learning. Same reason you change one variable per ablation run. Clean pairs = clean credit assignment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
print("repo root:", ROOT)

import json
swz = json.loads((ROOT / "data" / "normalized" / "swz_MUL0035.json").read_text())
hero = json.loads((ROOT / "data" / "normalized" / "hero_001.json").read_text())
print("a real corpus failure (the table found it): 0:07 on swz_MUL0035, 2,220ms overlap -")
for t in swz["turns"][:3]:
    print(f"  {t['turn_id']} {t['speaker']:>6}: {t['text'][:70]}")
print("\nPAUSE and look at the speakers. The USER barged in over the AGENT. Would you train")
print("the agent on this? NO - a user barge-in is a signal about the user's experience")
print("(impatience, eagerness), not an agent sin. We mine AGENT-side failures for pairs.")
print("Selecting which failures are trainable is itself a judgment call the pipeline encodes.")
print("\nThe hero call has the perfect agent-side specimen:")
for t in hero["turns"][1:3]:
    print(f"  {t['turn_id']} {t['speaker']:>6}: {t['text']}")
print("\nagent sin x2: barged in (timing) AND over-demanded instead of acknowledging (content).")

Now author the pair from the hero call's agent sin. Read each field and ask: could a gradient learn anything EXCEPT "acknowledge the partial answer, ask one follow-up"? That is the test.

In [ ]:
pair = {
    "prompt": [
        {"role": "system", "content": "Voice agent for appointment booking. Replies under 2 sentences. Never speak while the caller is mid-answer; acknowledge partial info before asking ONE follow-up."},
        {"role": "user", "content": hero["turns"][1]["text"]},
    ],
    "chosen":   [{"role": "assistant", "content": "Got it - Madhapur, near the metro station. Morning or evening slot work better?"}],
    "rejected": [{"role": "assistant", "content": hero["turns"][2]["text"]}],
}
print(json.dumps(pair, indent=2)[:800])

line = json.dumps(pair)
openai_line = json.dumps({"input": {"messages": pair["prompt"]},
                          "preferred_output": pair["chosen"],
                          "non_preferred_output": pair["rejected"]})
print("\nTRL jsonl bytes:", len(line), "| OpenAI-format bytes:", len(openai_line))

Two formats, one authored source — the 3-line mapper you just ran is the entire "OpenAI mirror" in Block 5. Provenance fields (`call_id`, `failure_dimension`, `needs_human_review: true`) ride along in our schema so every pair traces back to the exact ms-stamped failure that spawned it. `needs_human_review` defaults true because auto-generated chosen turns are *proposals* — honest pipelines say so.

## Exercise — your turn, second axis
Author a pair for a **latency** failure (agent took 1.6s of dead air, then answered fine). Careful — the failure is not the words, it is the silence. What does "chosen" even mean here? (Hint: think about what a *text* pair CAN and CANNOT teach. The checker prints a discussion.)

In [ ]:
my_pair = {
    "prompt": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}],
    "chosen": [{"role": "assistant", "content": "..."}],
    "rejected": [{"role": "assistant", "content": "..."}],
}
assert set(my_pair) == {"prompt", "chosen", "rejected"}
assert my_pair["chosen"] != my_pair["rejected"], "chosen and rejected must differ"
print("structure ok.\n")
print("Discussion: pure dead-air cannot be fixed by a text preference pair - silence is runtime behavior")
print("(endpointing/config), not token choice. What CAN be taught: filler acknowledgements like a brief")
print("'one moment' before slow lookups, or shorter turns that reduce processing time. This boundary -")
print("which failures are weight-fixable vs config-fixable - is a SENIOR-ENGINEER distinction. Use it in the room.")

## Prompt-level vs weight-level fixes (the moat sentence)
A detected failure can be fixed at three depths: **config** (endpointing ms, interruption thresholds), **prompt** (system-prompt rules — cheap, instant, ceiling-limited), or **weights** (DPO on accumulated pairs — durable, portable, compounding). Tomorrow's A/B block demonstrates the prompt-level loop end-to-end; the DPO queue is the weight-level asset you *own and take with you* — which is the answer when someone says "Leaping AI already closes the loop" (theirs is prompt-level, locked to their platform).

## Self-check
1. Recite the ladder: pretraining / SFT / preference tuning — what does each install?
2. DPO vs RLHF in one sentence each.
3. Why must chosen and rejected differ on exactly one axis?
4. Why does `needs_human_review` default to true?
5. Name a failure that should NOT become a DPO pair, and where its fix lives instead.

<details><summary>Answers</summary>

1. Capability (predict text) / format-and-style imitation from curated demos / preferring better over worse behavior from pairs.
2. RLHF: fit a reward model on pairs then RL against it (two stages, unstable, powerful). DPO: algebraic shortcut that optimizes the policy directly on pairs against a frozen reference (one stage, stable, standard tooling).
3. Credit assignment — multi-axis diffs leave the gradient ambiguous about what is being preferred; single-variable changes, like ablations.
4. The chosen turns are machine-proposed corrections; shipping them as ground truth without human eyes would be silent label noise — and dishonest.
5. Pure latency / dead air (or barge-in *timing* itself) — runtime/config territory: endpointing hold, interruption handling, infra speed. Text pairs teach token choices, not clocks.
</details>